# Eye State classifier — train บน MRL Eye Dataset

โมเดลตัดสินว่าตาใน **Eye Region** เปิดหรือปิด ให้ค่าออกเป็น **P(ตาปิด)** ใช้บน ESP32-S3 ด้วย TFLite Micro (INT8)

ขั้นตอนใน notebook นี้
1. ดาวน์โหลด MRL Eye Dataset → อ่าน label จากชื่อไฟล์
2. แบ่ง train / val / test **ตาม subject** (คนใน test ไม่เคยถูกเห็นตอน train)
3. preprocess ด้วย `preprocess.py` (ตัวเดียวกับ `eye_preprocess.h` บนบอร์ด, ตรวจ bit-exact ด้วย `test_preprocess.py`)
4. train CNN เล็กแบบเดียวกันที่ขนาดภาพเข้า 32 / 48 / 64 แล้วเทียบความแม่นกับขนาดโมเดล
5. export INT8 full-integer → `.tflite` → `model_data.h` + `golden_data.h` สำหรับ golden test บนบอร์ด

**รันบน Colab:** Runtime → Run all (GPU ไม่จำเป็นแต่เร็วกว่า) · cell แรก clone repo ให้เองถ้ายังไม่มี `preprocess.py`
**รันในเครื่อง:** เปิดจากโฟลเดอร์ `ml/` (ดู `README.md`)

ผลทั้งหมดเขียนลง `ml/output/` · dataset อยู่ที่ `ml/data/` (ทั้งสองโฟลเดอร์ไม่ commit)

In [ ]:
import os, sys, pathlib, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Franknakub/driver-drowsiness-detection.git"
REPO_BRANCH = "firmware"   # branch ที่มี ml/preprocess.py

if not pathlib.Path("preprocess.py").exists():
    if IN_COLAB:
        if not pathlib.Path("driver-drowsiness-detection").exists():
            subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL], check=True)
        os.chdir("driver-drowsiness-detection/ml")
ML_DIR = pathlib.Path.cwd()
assert (ML_DIR / "preprocess.py").exists(), "ต้องรันจากโฟลเดอร์ ml/ (หรืออัปโหลด preprocess.py ขึ้น Colab)"
print("ML_DIR =", ML_DIR, "| Colab:", IN_COLAB)

In [ ]:
import io, json, math, time, zipfile, urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

import preprocess as pp

SIZES = [32, 48, 64]      # ขนาดภาพเข้าที่จะเทียบ
SEED = 42
BATCH = 128
EPOCHS = 30               # สูงสุด; EarlyStopping หยุดก่อนได้
TEST_FRAC = 0.15          # สัดส่วนภาพ (ไม่ใช่สัดส่วนคน) ที่ให้ test / val
VAL_FRAC = 0.15
LOWRES_RANGE = (14, 30)   # จำลอง Eye Region จริงบน QVGA ที่กว้าง ~20-30 px
GOLDEN_CROP = 24          # ขนาด crop จำลองสำหรับ golden test
GOLDEN_COUNT = 16

DATA_DIR = ML_DIR / "data"
OUT_DIR = ML_DIR / "output"
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

MRL_URL = "https://mrl.cs.vsb.cz/data/eyedataset/mrlEyes_2018_01.zip"
MRL_ZIP_BYTES = 341_866_898   # ขนาดไฟล์จริง (ตรวจ 17 ต.ค. 2026)

QUICK = os.environ.get("EYE_QUICK") == "1"   # smoke test: 1 epoch, train subset
if QUICK:
    EPOCHS = 1

keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. ดาวน์โหลด MRL Eye Dataset

หน้าเว็บ: <https://mrl.cs.vsb.cz/eyedataset.html> · ไฟล์ `mrlEyes_2018_01.zip` (342 MB, 84,898 ภาพ PNG grayscale, 37 คน)
ถ้าใช้ในรายงานต้องอ้าง R. Fusek, *Pupil localization using geodesic distance*, ISVC 2018

In [ ]:
zip_path = DATA_DIR / "mrlEyes_2018_01.zip"
mrl_dir = DATA_DIR / "mrlEyes_2018_01"

if not zip_path.exists() or zip_path.stat().st_size != MRL_ZIP_BYTES:
    print("downloading", MRL_URL)
    tmp = zip_path.with_suffix(".part")
    urllib.request.urlretrieve(MRL_URL, tmp)
    assert tmp.stat().st_size == MRL_ZIP_BYTES, f"ขนาดไม่ตรง: {tmp.stat().st_size}"
    tmp.replace(zip_path)
if not mrl_dir.exists():
    print("extracting ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
print(zip_path.stat().st_size, "bytes OK")

## 2. อ่าน label จากชื่อไฟล์

`s0001_00001_0_0_0_0_0_01.png` = `subject_image_gender_glasses_eyestate_reflections_lighting_sensor`
`eyestate` 0 = ปิด, 1 = เปิด · target ของโมเดลคือ `closed = 1 - eyestate` (โมเดลตอบ P(ตาปิด))

In [ ]:
FIELDS = ["subject", "image_id", "gender", "glasses", "eye_state", "reflections", "lighting", "sensor"]
files = sorted(mrl_dir.glob("s*/*.png"))
rows = []
for f in files:
    parts = f.stem.split("_")
    assert len(parts) == 8, f.name
    rows.append(dict(zip(FIELDS, parts), path=str(f)))
meta = pd.DataFrame(rows)
for c in ["image_id", "gender", "glasses", "eye_state", "reflections", "lighting"]:
    meta[c] = meta[c].astype(int)
meta["closed"] = 1 - meta["eye_state"]
assert meta["eye_state"].isin([0, 1]).all()

print(len(meta), "images,", meta.subject.nunique(), "subjects")
print("closed:", meta.closed.sum(), "| open:", (1 - meta.closed).sum())
print("glasses:", meta.glasses.mean().round(3), "| sensors:", meta.sensor.value_counts().to_dict())

## 3. แบ่ง train / val / test ตาม subject

สุ่มลำดับคนด้วย seed คงที่ แล้วเติมคนเข้า test จนได้ ≥15% ของภาพ ต่อด้วย val อีก ≥15% ที่เหลือเป็น train
**ห้ามสุ่มรายภาพ** เพราะภาพของคนเดียวกันจากวิดีโอเดียวกันคล้ายกันมาก ความแม่นจะสูงเกินจริง

In [ ]:
rng = np.random.default_rng(SEED)
subjects = sorted(meta.subject.unique())
rng.shuffle(subjects)
counts = meta.subject.value_counts()
total = len(meta)
split_of, filled = {}, {"test": 0, "val": 0}
for s in subjects:
    if filled["test"] < TEST_FRAC * total:
        split_of[s] = "test"; filled["test"] += counts[s]
    elif filled["val"] < VAL_FRAC * total:
        split_of[s] = "val"; filled["val"] += counts[s]
    else:
        split_of[s] = "train"
meta["split"] = meta.subject.map(split_of)

summary = meta.groupby("split").agg(
    subjects=("subject", "nunique"), images=("path", "size"),
    closed_frac=("closed", "mean"), glasses_frac=("glasses", "mean"))
summary["image_frac"] = summary.images / total
print(summary.round(3))
for sp in ["train", "val", "test"]:
    print(sp, sorted(meta.loc[meta.split == sp, "subject"].unique()))
assert set(meta[meta.split == "train"].subject).isdisjoint(meta[meta.split == "test"].subject)
pd.Series(split_of, name="split").sort_index().to_csv(OUT_DIR / "subject_split.csv")

## 4. Preprocess

ทุกภาพผ่าน `pp.resize_bilinear` (integer bilinear, half-pixel centers) เป็น N×N ครั้งเดียวแล้ว cache ไว้
ตอน train ป้อนเป็น float `gray / 255` ซึ่งหลัง quantize INT8 จะเท่ากับ `gray - 128` พอดีบนบอร์ด

In [ ]:
cache_path = DATA_DIR / ("cache_mrl_" + "_".join(map(str, SIZES)) + ".npz")
names = np.array([pathlib.Path(p).name for p in meta.path])

if cache_path.exists() and np.array_equal(np.load(cache_path)["names"], names):
    with np.load(cache_path) as d:
        X = {n: d[f"x{n}"] for n in SIZES}
    print("loaded cache", cache_path.name)
else:
    X = {n: np.empty((len(meta), n, n), np.uint8) for n in SIZES}
    t0 = time.time()
    for i, p in enumerate(meta.path):
        g = np.asarray(Image.open(p).convert("L"))
        for n in SIZES:
            X[n][i] = pp.resize_bilinear(g, n)
        if i % 20000 == 0:
            print(f"{i}/{len(meta)}  {time.time() - t0:.0f}s")
    np.savez(cache_path, names=names, **{f"x{n}": X[n] for n in SIZES})
    print(f"preprocessed in {time.time() - t0:.0f}s")

y = meta.closed.values.astype(np.float32)
idx = {sp: np.flatnonzero(meta.split.values == sp) for sp in ["train", "val", "test"]}
if QUICK:
    idx = {k: np.random.default_rng(SEED).choice(v, 3000, replace=False) for k, v in idx.items()}
print({k: len(v) for k, v in idx.items()})

In [ ]:
# ตัวอย่างภาพหลัง preprocess: แถวบน = เปิด, แถวล่าง = ปิด
fig, axes = plt.subplots(2 * len(SIZES), 10, figsize=(12, 2.6 * len(SIZES)))
pick_rng = np.random.default_rng(0)
for r, n in enumerate(SIZES):
    for c, cls in enumerate([0, 1]):
        sel = pick_rng.choice(np.flatnonzero(y == cls), 10, replace=False)
        for k, j in enumerate(sel):
            ax = axes[2 * r + c, k]
            ax.imshow(X[n][j], cmap="gray", vmin=0, vmax=255)
            ax.set_xticks([]); ax.set_yticks([])
        axes[2 * r + c, 0].set_ylabel(f"{n}px {'closed' if cls else 'open'}", fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "samples.png", dpi=80); plt.show()

## 5. Augmentation

ทำเฉพาะชุด train บน float [0, 1] (ไม่ต้องทำซ้ำบนบอร์ด)
- **เลื่อน/ซูมกรอบ** ±8% / −15…+10% — Eye Region จากจุดตาของ ESP-DL ไม่ได้อยู่ตรงกลางเป๊ะ
- **พลิกซ้ายขวา** — ตาซ้าย/ขวาใช้โมเดลเดียวกัน
- **ความละเอียดต่ำ** 14–30 px แล้วขยายกลับ (50%) — ตาจริงใน QVGA กว้างแค่ ~20–30 px ขณะที่ภาพ MRL ส่วนใหญ่ 60–130 px
- **blur** Gaussian σ 0.7 / 1.3 (สุ่ม 1 ใน 3 ไม่ blur)
- **gamma, contrast, ความสว่าง, noise** — แสงในรถและ gain ของ OV5640
- ปัดค่ากลับเป็นขั้น 1/255 เหมือนภาพ uint8 จริง

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
jitter = keras.Sequential([
    layers.RandomTranslation(0.08, 0.08, fill_mode="reflect", seed=SEED),
    layers.RandomZoom((-0.15, 0.10), fill_mode="reflect", seed=SEED + 1),
])

def _gauss_kernel(sigma, k=5):
    ax = np.arange(k) - k // 2
    g = np.exp(-(ax ** 2) / (2 * sigma ** 2))
    g = np.outer(g, g); g /= g.sum()
    return tf.constant(g[:, :, None, None], tf.float32)

BLUR_KERNELS = [_gauss_kernel(0.7), _gauss_kernel(1.3)]

def augment(x, y):
    b = tf.shape(x)[0]
    n = x.shape[1]
    u = lambda lo, hi: tf.random.uniform([b, 1, 1, 1], lo, hi)
    x = jitter(x, training=True)
    x = tf.where(u(0, 1) < 0.5, tf.reverse(x, axis=[2]), x)
    s = tf.random.uniform([], LOWRES_RANGE[0], min(LOWRES_RANGE[1], n) + 1, tf.int32)
    lo = tf.image.resize(tf.image.resize(x, [s, s], "bilinear", antialias=True), [n, n], "bilinear")
    x = tf.where(u(0, 1) < 0.5, lo, x)
    pad = tf.pad(x, [[0, 0], [2, 2], [2, 2], [0, 0]], mode="REFLECT")
    blur1 = tf.nn.conv2d(pad, BLUR_KERNELS[0], 1, "VALID")
    blur2 = tf.nn.conv2d(pad, BLUR_KERNELS[1], 1, "VALID")
    r = u(0, 1)
    x = tf.where(r < 1 / 3, blur1, tf.where(r < 2 / 3, blur2, x))
    x = tf.pow(tf.clip_by_value(x, 1e-6, 1.0), tf.exp(u(math.log(0.7), math.log(1.4))))
    mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
    x = (x - mean) * u(0.75, 1.3) + mean + u(-0.1, 0.1)
    x = x + tf.random.normal(tf.shape(x)) * u(0.0, 0.02)
    x = tf.round(tf.clip_by_value(x, 0.0, 1.0) * 255.0) / 255.0
    return x, y

def make_ds(n, split, train=False):
    xs, ys = X[n][idx[split]][..., None], y[idx[split]]
    ds = tf.data.Dataset.from_tensor_slices((xs, ys))
    if train:
        ds = ds.shuffle(len(xs), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(lambda a, b: (tf.cast(a, tf.float32) / 255.0, b), num_parallel_calls=AUTOTUNE)
    if train:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

# ดูผล augmentation
xa, _ = augment(tf.constant(X[64][idx["train"][:16]][..., None] / 255.0, tf.float32), tf.zeros(16))
fig, axes = plt.subplots(2, 16, figsize=(16, 2.2))
for k in range(16):
    axes[0, k].imshow(X[64][idx["train"][k]], cmap="gray", vmin=0, vmax=255)
    axes[1, k].imshow(xa[k, :, :, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.suptitle("top: original 64px / bottom: augmented"); plt.savefig(OUT_DIR / "augment.png", dpi=80); plt.show()

## 6. โมเดล

CNN เล็กโครงเดียวกันทุกขนาด: 3 ชั้น `Conv3×3 → BN → ReLU → MaxPool` (16/32/64 filter) → GlobalAveragePooling → Dropout → Dense(1, sigmoid)
ทุก op (CONV_2D, MAX_POOL_2D, MEAN, FULLY_CONNECTED, LOGISTIC) มีใน TFLite Micro · BN ถูกรวมเข้า conv ตอน convert

In [ ]:
def build_model(n):
    inp = keras.Input((n, n, 1), name="eye")
    x = inp
    for f in (16, 32, 64):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid", name="p_closed")(x)
    return keras.Model(inp, out, name=f"eye_state_{n}")

def count_macs(model):
    macs = 0
    for l in model.layers:
        if isinstance(l, layers.Conv2D):
            _, h, w, c = l.output.shape
            kh, kw = l.kernel_size
            macs += h * w * c * kh * kw * l.input.shape[-1]
        elif isinstance(l, layers.Dense):
            macs += l.input.shape[-1] * l.units
    return macs

build_model(64).summary()

## 7. Train ทั้ง 3 ขนาด

Adam 1e-3 · binary cross-entropy · ถ่วง class ตามสัดส่วนใน train · EarlyStopping ที่ val_loss (patience 4, คืน weight ที่ดีที่สุด) · ลด LR เมื่อ val_loss ไม่ลด

In [ ]:
def metrics_from(p, t, thr=0.5):
    pred = (p >= thr).astype(int); t = t.astype(int)
    tp = int(((pred == 1) & (t == 1)).sum()); tn = int(((pred == 0) & (t == 0)).sum())
    fp = int(((pred == 1) & (t == 0)).sum()); fn = int(((pred == 0) & (t == 1)).sum())
    rec = tp / max(tp + fn, 1); prec = tp / max(tp + fp, 1)
    return dict(accuracy=(tp + tn) / len(t), closed_recall=rec, closed_precision=prec,
                closed_f1=2 * prec * rec / max(prec + rec, 1e-9), open_recall=tn / max(tn + fp, 1),
                tp=tp, tn=tn, fp=fp, fn=fn)

pos = y[idx["train"]].mean()
class_weight = {0: 0.5 / (1 - pos), 1: 0.5 / pos}
print("train closed fraction", round(float(pos), 3), "class_weight", class_weight)

models, histories, results = {}, {}, {}
for n in SIZES:
    keras.utils.set_random_seed(SEED)
    model = build_model(n)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.Recall(name="closed_recall"), keras.metrics.AUC(name="auc")])
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-5)]
    t0 = time.time()
    h = model.fit(make_ds(n, "train", True), validation_data=make_ds(n, "val"), epochs=EPOCHS,
                  class_weight=class_weight, callbacks=cb, verbose=2)
    train_s = time.time() - t0
    p = model.predict(make_ds(n, "test"), verbose=0)[:, 0]
    res = metrics_from(p, y[idx["test"]])
    res.update(params=model.count_params(), macs=count_macs(model), epochs=len(h.history["loss"]),
               train_seconds=round(train_s), auc=float(keras.metrics.AUC()(y[idx["test"]], p)))
    models[n], histories[n], results[n] = model, h.history, {"float": res}
    (OUT_DIR / f"size_{n}").mkdir(exist_ok=True)
    model.save(OUT_DIR / f"size_{n}" / "model_float.keras")
    print(f"== {n}px  test acc {res['accuracy']:.4f}  closed recall {res['closed_recall']:.4f}  "
          f"({res['epochs']} epochs, {train_s:.0f}s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for n in SIZES:
    axes[0].plot(histories[n]["val_loss"], label=f"{n}px val"); axes[0].plot(histories[n]["loss"], "--", alpha=.5, label=f"{n}px train")
    axes[1].plot(histories[n]["val_accuracy"], label=f"{n}px")
axes[0].set_title("loss"); axes[1].set_title("val accuracy"); axes[0].legend(fontsize=7); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "curves.png", dpi=80); plt.show()

## 8. Export INT8 full-integer

- representative dataset 500 ภาพจาก train + ภาพครึ่งดำครึ่งขาว 1 ภาพ เพื่อให้ช่วง input เป็น [0, 1] พอดี → input scale = 1/255, zero_point = −128 → **บนบอร์ด `int8 = gray − 128`**
- input/output เป็น int8 · output ของ sigmoid ใน TFLite มี scale 1/256, zero_point −128 → **P(ตาปิด) = (q + 128) / 256**
- วัดความแม่นของโมเดล INT8 จริงบน test ทั้งชุดด้วย TFLite interpreter

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")

def get_interpreter(model_content, reference=False):
    # no XNNPACK delegate, so ops/accuracy reflect the plain int8 kernels;
    # reference=True uses the reference kernels (closest to TFLite Micro)
    try:
        from ai_edge_litert import interpreter as lit
        Interpreter, Resolver = lit.Interpreter, lit.OpResolverType
    except ImportError:
        Interpreter, Resolver = tf.lite.Interpreter, tf.lite.experimental.OpResolverType
    kind = Resolver.BUILTIN_REF if reference else Resolver.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    return Interpreter(model_content=model_content, experimental_op_resolver_type=kind)

def convert_int8(model, n):
    sm_dir = OUT_DIR / f"size_{n}" / "saved_model"
    model.export(str(sm_dir), format="tf_saved_model", verbose=False)
    rep_idx = np.random.default_rng(SEED).choice(idx["train"], 500, replace=False)
    def rep():
        edge = np.zeros((1, n, n, 1), np.float32); edge[:, :, n // 2:] = 1.0
        yield [edge]
        for j in rep_idx:
            yield [X[n][j][None, ..., None].astype(np.float32) / 255.0]
    conv = tf.lite.TFLiteConverter.from_saved_model(str(sm_dir))
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    return conv.convert()

def run_int8(tfl, x_uint8, batch=256, reference=False):
    # x_uint8: (M, n, n) gray -> int8 output q  (input quantized exactly like eye_to_int8)
    it = get_interpreter(tfl, reference)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    n = x_uint8.shape[1]
    qs = []
    for s in range(0, len(x_uint8), batch):
        xb = pp.to_int8(x_uint8[s:s + batch])[..., None]
        it.resize_tensor_input(inp["index"], [len(xb), n, n, 1]); it.allocate_tensors()
        it.set_tensor(inp["index"], xb); it.invoke()
        qs.append(it.get_tensor(out["index"])[:, 0].copy())
    return np.concatenate(qs)

def tflite_info(tfl):
    it = get_interpreter(tfl); it.allocate_tensors()
    ops = it._get_ops_details()
    tensors = {t["index"]: t for t in it.get_tensor_details()}
    act = {o for op in ops for o in op["outputs"]} | {it.get_input_details()[0]["index"]}
    nbytes = lambda i: int(np.prod(tensors[i]["shape"])) * np.dtype(tensors[i]["dtype"]).itemsize
    peak = max(sum(nbytes(i) for i in list(op["inputs"]) + list(op["outputs"]) if i in act) for op in ops)
    return dict(ops=sorted({op["op_name"] for op in ops}), peak_activation_bytes=peak,
                input=it.get_input_details()[0]["quantization"], output=it.get_output_details()[0]["quantization"])

tflites = {}
for n in SIZES:
    tfl = convert_int8(models[n], n)
    tflites[n] = tfl
    (OUT_DIR / f"size_{n}" / "model_int8.tflite").write_bytes(tfl)
    info = tflite_info(tfl)
    in_scale, in_zp = info["input"]; out_scale, out_zp = info["output"]
    assert abs(in_scale * 255 - 1) < 1e-6 and in_zp == -128, info["input"]
    assert abs(out_scale * 256 - 1) < 1e-6 and out_zp == -128, info["output"]
    q = run_int8(tfl, X[n][idx["test"]])
    p8 = (q.astype(np.float32) - out_zp) * out_scale
    res8 = metrics_from(p8, y[idx["test"]])
    res8.update(tflite_bytes=len(tfl), peak_activation_bytes=info["peak_activation_bytes"], ops=info["ops"],
                auc=float(keras.metrics.AUC()(y[idx["test"]], p8)))
    results[n]["int8"] = res8
    print(f"{n}px  int8 acc {res8['accuracy']:.4f}  closed recall {res8['closed_recall']:.4f}  "
          f"{len(tfl)} B  ops {info['ops']}")

## 9. เทียบขนาดภาพเข้า

ทุกตัวเลขวัดบนชุด **test (คนที่ไม่เคยเห็น)** ที่ threshold 0.5 · `peak act` = activation in+out ที่ใหญ่สุดของ op เดียว (ประมาณขั้นต่ำของ tensor arena; ค่าจริงต้องวัดบนบอร์ดด้วย `arena_used_bytes()`)

In [ ]:
table = pd.DataFrame([{
    "input": f"{n}x{n}",
    "float acc": results[n]["float"]["accuracy"],
    "float closed recall": results[n]["float"]["closed_recall"],
    "int8 acc": results[n]["int8"]["accuracy"],
    "int8 closed recall": results[n]["int8"]["closed_recall"],
    "int8 closed precision": results[n]["int8"]["closed_precision"],
    "int8 AUC": results[n]["int8"]["auc"],
    "params": results[n]["float"]["params"],
    "MACs": results[n]["float"]["macs"],
    "tflite KB": results[n]["int8"]["tflite_bytes"] / 1024,
    "peak act KB": results[n]["int8"]["peak_activation_bytes"] / 1024,
    "epochs": results[n]["float"]["epochs"],
} for n in SIZES])
table.to_csv(OUT_DIR / "size_comparison.csv", index=False)
print(table.round(4).to_string(index=False))
json.dump({str(n): results[n] for n in SIZES}, open(OUT_DIR / "results.json", "w"), indent=2)

In [ ]:
# ความแม่น INT8 แยกตามกลุ่มย่อยใน test (รายคน / แว่น / แสงสะท้อน / แสง / เซนเซอร์)
test_meta = meta.iloc[idx["test"]].reset_index(drop=True)
rows = []
for n in SIZES:
    it = get_interpreter(tflites[n]); out_scale, out_zp = it.get_output_details()[0]["quantization"]
    p8 = (run_int8(tflites[n], X[n][idx["test"]]).astype(np.float32) - out_zp) * out_scale
    for col in ["subject", "glasses", "reflections", "lighting", "sensor"]:
        for v, g in test_meta.groupby(col):
            m = metrics_from(p8[g.index], g.closed.values)
            rows.append({"input": n, "group": f"{col}={v}", "images": len(g),
                         "acc": m["accuracy"], "closed_recall": m["closed_recall"]})
groups = pd.DataFrame(rows).pivot_table(index=["group", "images"], columns="input", values=["acc", "closed_recall"])
groups.to_csv(OUT_DIR / "test_by_group.csv")
print(groups.round(3).to_string())

## 10. `model_data.h` + golden test

สำหรับทุกขนาดเขียนลง `output/size_N/`
- `model_data.h` — โมเดล INT8 เป็น C array (`alignas(16)`) + ค่า quantization
- `golden_data.h` — ภาพ test 16 ภาพ (ปิด 8 เปิด 8) ย่อเป็น crop gray 24×24 (ขนาดราว Eye Region จริงบน QVGA) + ผลที่ PC คาดไว้
  บนบอร์ด: `eye_resize_bilinear(crop 24×24 → N×N)` → `eye_to_int8` → invoke → เทียบ checksum ของ input และค่า output q
  (input ต้องตรงทุก byte · output ยอมให้ต่าง ±10 LSB ≈ ±0.04 ของ P เพราะ logit ที่ปัดเศษต่างกัน 1 LSB จะขยายเป็น 6–9 LSB หลัง LOGISTIC ช่วง P ≈ 0.5 — วัดจาก kernel reference กับ optimized บน PC เอง)
- `expected_q` ใช้ kernel **reference** ของ TFLite (ใกล้ TFLite Micro ที่สุด)

In [ ]:
def c_bytes(data, per_line=16):
    data = bytes(data)
    return "\n".join("  " + ", ".join(f"0x{b:02x}" for b in data[i:i + per_line]) + ","
                     for i in range(0, len(data), per_line))

def fnv1a(data):
    h = 0x811C9DC5
    for b in bytes(data):
        h = ((h ^ b) * 0x01000193) & 0xFFFFFFFF
    return h

def write_model_h(n, tfl, path):
    it = get_interpreter(tfl)
    (in_s, in_zp), (out_s, out_zp) = it.get_input_details()[0]["quantization"], it.get_output_details()[0]["quantization"]
    path.write_text(f'''// Generated by ml/train.ipynb -- eye-state classifier (MRL Eye Dataset), INT8 full-integer.
// Input : int8 [1,{n},{n},1] = gray - 128  (scale {in_s:.9g}, zero_point {in_zp})
// Output: int8 [1,1] -> P(closed) = (q - ({out_zp})) * {out_s:.9g}
// Ops   : {", ".join(tflite_info(tfl)["ops"])}
#pragma once
#include <stdint.h>

#define EYE_MODEL_INPUT_SIZE {n}
#define EYE_MODEL_INPUT_ZERO_POINT ({in_zp})
#define EYE_MODEL_OUTPUT_SCALE ({out_s:.9g}f)
#define EYE_MODEL_OUTPUT_ZERO_POINT ({out_zp})

alignas(16) const uint8_t g_eye_model_data[] = {{
{c_bytes(tfl)}
}};
const unsigned int g_eye_model_data_len = {len(tfl)};
''', encoding="utf-8")

grng = np.random.default_rng(SEED)
golden_idx = np.concatenate([grng.choice(idx["test"][y[idx["test"]] == c], GOLDEN_COUNT // 2, replace=False) for c in (1, 0)])
golden_crops = np.stack([pp.resize_bilinear(np.asarray(Image.open(meta.path[j]).convert("L")), GOLDEN_CROP) for j in golden_idx])

def write_golden_h(n, tfl, path):
    inputs = np.stack([pp.preprocess_gray(c, n) for c in golden_crops])
    small = np.stack([pp.resize_bilinear(c, n) for c in golden_crops])
    q = run_int8(tfl, small, reference=True)
    q_opt = run_int8(tfl, small)
    print(f"{n}px golden: max |reference - optimized| = {int(np.abs(q.astype(int) - q_opt.astype(int)).max())} LSB")
    it = get_interpreter(tfl); out_s, out_zp = it.get_output_details()[0]["quantization"]
    lines = [f"// Generated by ml/train.ipynb -- golden test for the {n}x{n} eye-state model.",
             f"// crop {GOLDEN_CROP}x{GOLDEN_CROP} gray -> eye_resize_bilinear -> eye_to_int8 -> invoke.",
             "// input_fnv1a = FNV-1a 32-bit over the int8 input tensor bytes; expected_q = PC TFLite output.",
             "#pragma once", "#include <stdint.h>", "",
             f"#define GOLDEN_COUNT {len(golden_crops)}", f"#define GOLDEN_CROP {GOLDEN_CROP}", "",
             "const uint8_t g_golden_crops[GOLDEN_COUNT][GOLDEN_CROP * GOLDEN_CROP] = {"]
    for c in golden_crops:
        lines += ["  {", c_bytes(c.tobytes()), "  },"]
    lines += ["};", "",
              "const uint8_t g_golden_label_closed[GOLDEN_COUNT] = {" + ", ".join(str(int(y[j])) for j in golden_idx) + "};",
              "const uint32_t g_golden_input_fnv1a[GOLDEN_COUNT] = {" + ", ".join(f"0x{fnv1a(i.tobytes()):08x}u" for i in inputs) + "};",
              "const int8_t g_golden_expected_q[GOLDEN_COUNT] = {" + ", ".join(str(int(v)) for v in q) + "};",
              "// P(closed): " + ", ".join(f"{(int(v) - out_zp) * out_s:.3f}" for v in q), ""]
    path.write_text("\n".join(lines), encoding="utf-8")
    return q

for n in SIZES:
    d = OUT_DIR / f"size_{n}"
    write_model_h(n, tflites[n], d / "model_data.h")
    q = write_golden_h(n, tflites[n], d / "golden_data.h")
    print(f"{n}px: model_data.h {(d / 'model_data.h').stat().st_size // 1024} KB | golden q {q.tolist()}")

## 11. ตรวจ `model_data.h` ย้อนกลับ

อ่าน byte จาก header กลับมาแล้วต้องได้ไฟล์ `.tflite` เดิมทุก byte

In [ ]:
import re
for n in SIZES:
    txt = (OUT_DIR / f"size_{n}" / "model_data.h").read_text()
    body = txt.split("g_eye_model_data[] = {")[1].split("};")[0]
    back = bytes(int(h, 16) for h in re.findall(r"0x([0-9a-f]{2})", body))
    assert back == tflites[n], n
print("model_data.h round-trip OK for", SIZES)
print("outputs:", sorted(str(p.relative_to(OUT_DIR)) for p in OUT_DIR.rglob("*") if p.is_file() and "saved_model" not in str(p)))